### Simple GenAI App Using LangChain And OpenAI Model

In [ ]:
import os
from dotenv import load_dotenv

# Environment Variables
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") 
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# Data Ingestion
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
docs = loader.load()
docs

c:\Users\viren\anaconda3\envs\langchain_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# Data Transformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
documents

In [ ]:
# Embeddings 
from langchain_openai.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
# VectorStore
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(documents, embeddings)
vector_db.save_local("faiss_index")
vector_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

In [ ]:
# Retriever
retriever = vector_db.as_retriever()

In [ ]:
# OpenAI LLM
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-4o')

In [ ]:
# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert AI assistant. 
     Answer the question based ONLY on the provided context. Context: {context}"""),
    ("user", "{input}")
])

prompt

In [ ]:
# Output Parser 
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [ ]:
# LCEL Chain
from langchain_core.runnables import RunnablePassthrough

chain = ( {"context": retriever, "input": RunnablePassthrough()} | prompt | model | output_parser)

In [ ]:
# Query
input_query = "Summarize the document"
response = chain.invoke(input_query)
response